In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# Task 1: Exploratory Data Analysis\n", "## AlphaCare Insurance Solutions — Risk Analytics\n", "**Analyst:** Omega Consultancy | **Data:** Feb 2014 – Aug 2015"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import matplotlib\n",
    "import seaborn as sns\n",
    "import warnings\n",
    "import sys\n",
    "sys.path.append('..')\n",
    "from src.data_loader import load_data, get_basic_info, add_derived_metrics\n",
    "from src.eda_utils import plot_distributions, plot_correlation, plot_geographic, plot_outliers\n",
    "warnings.filterwarnings('ignore')\n",
    "matplotlib.use('Agg')\n",
    "sns.set_theme(style='whitegrid', palette='muted')\n",
    "plt.rcParams['figure.dpi'] = 120\n",
    "print('Libraries loaded.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 1. Load Data"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df = load_data()\n",
    "df = add_derived_metrics(df)\n",
    "print(df.shape)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 2. Data Summarization"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "missing = get_basic_info(df)\n",
    "print('\\nNumerical summary:')\n",
    "df[['TotalPremium','TotalClaims','LossRatio','Margin','CustomValueEstimate']].describe().round(2)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 3. Data Quality Assessment"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print('Missing values summary:')\n",
    "missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)\n",
    "missing_pct = missing_pct[missing_pct > 0]\n",
    "print(missing_pct.head(20))\n",
    "print(f'\\nTotal columns with missing values: {len(missing_pct)}')\n",
    "print('\\nHandling strategy:')\n",
    "print('  - Columns >50% missing: drop')\n",
    "print('  - Numerical <50% missing: fill with median')\n",
    "print('  - Categorical <50% missing: fill with mode')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 4. Guiding Question 1: Overall Loss Ratio"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "overall_lr = df['TotalClaims'].sum() / df['TotalPremium'].sum()\n",
    "print(f'Overall Loss Ratio: {overall_lr:.4f} ({overall_lr*100:.2f}%)')\n",
    "print('\\nLoss Ratio by Province:')\n",
    "prov_lr = df.groupby('Province').apply(lambda x: x['TotalClaims'].sum()/x['TotalPremium'].sum()).sort_values(ascending=False)\n",
    "print(prov_lr.round(4))\n",
    "print('\\nLoss Ratio by VehicleType:')\n",
    "veh_lr = df.groupby('VehicleType').apply(lambda x: x['TotalClaims'].sum()/x['TotalPremium'].sum()).sort_values(ascending=False)\n",
    "print(veh_lr.head(10).round(4))\n",
    "print('\\nLoss Ratio by Gender:')\n",
    "gen_lr = df.groupby('Gender').apply(lambda x: x['TotalClaims'].sum()/x['TotalPremium'].sum())\n",
    "print(gen_lr.round(4))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 5. Plot 1: Loss Ratio by Province (Creative Visualization)"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(1, 2, figsize=(16, 6))\n",
    "fig.suptitle('Insurance Portfolio Risk Overview by Province', fontsize=14, fontweight='bold')\n",
    "\n",
    "prov_data = df.groupby('Province').agg(\n",
    "    TotalPremium=('TotalPremium','sum'),\n",
    "    TotalClaims=('TotalClaims','sum'),\n",
    "    LossRatio=('LossRatio','mean'),\n",
    "    PolicyCount=('PolicyID','count')\n",
    ").reset_index().sort_values('LossRatio', ascending=True)\n",
    "\n",
    "colors = ['#2ecc71' if lr < 0.5 else '#e74c3c' if lr > 1 else '#f39c12' for lr in prov_data['LossRatio']]\n",
    "bars = axes[0].barh(prov_data['Province'], prov_data['LossRatio'], color=colors)\n",
    "axes[0].axvline(x=1.0, color='red', linestyle='--', alpha=0.7, label='Break-even (LR=1.0)')\n",
    "axes[0].axvline(x=overall_lr, color='blue', linestyle='--', alpha=0.7, label=f'Portfolio avg ({overall_lr:.2f})')\n",
    "axes[0].set_xlabel('Loss Ratio')\n",
    "axes[0].set_title('Loss Ratio by Province')\n",
    "axes[0].legend(fontsize=8)\n",
    "for bar, val in zip(bars, prov_data['LossRatio']):\n",
    "    axes[0].text(val+0.01, bar.get_y()+bar.get_height()/2, f'{val:.2f}', va='center', fontsize=8)\n",
    "\n",
    "axes[1].scatter(prov_data['TotalPremium']/1e6, prov_data['TotalClaims']/1e6,\n",
    "                s=prov_data['PolicyCount']/50, c=prov_data['LossRatio'], cmap='RdYlGn_r', alpha=0.8)\n",
    "for _, row in prov_data.iterrows():\n",
    "    axes[1].annotate(row['Province'][:8], (row['TotalPremium']/1e6, row['TotalClaims']/1e6), fontsize=7)\n",
    "axes[1].set_xlabel('Total Premium (ZAR millions)')\n",
    "axes[1].set_ylabel('Total Claims (ZAR millions)')\n",
    "axes[1].set_title('Premium vs Claims by Province\\n(bubble size = policy count)')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/plot1_loss_ratio_province.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()\n",
    "print('Plot 1 saved.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 6. Guiding Question 2: Financial Variable Distributions"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(2, 3, figsize=(18, 10))\n",
    "fig.suptitle('Distribution of Key Financial Variables', fontsize=14, fontweight='bold')\n",
    "\n",
    "cols = ['TotalPremium','TotalClaims','LossRatio','Margin','CustomValueEstimate','SumInsured']\n",
    "for ax, col in zip(axes.flatten(), cols):\n",
    "    data = df[col].dropna()\n",
    "    data_clipped = data[data.between(data.quantile(0.01), data.quantile(0.99))]\n",
    "    ax.hist(data_clipped, bins=50, color='#3498db', alpha=0.7, edgecolor='white')\n",
    "    ax.set_title(f'{col}\\n(mean={data.mean():.0f}, std={data.std():.0f})')\n",
    "    ax.set_xlabel('Value (ZAR)')\n",
    "    ax.set_ylabel('Count')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/plot2_distributions.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()\n",
    "print('Plot 2 saved.')\n",
    "\n",
    "print('\\nOutlier check (values above 99th percentile):')\n",
    "for col in ['TotalPremium','TotalClaims','CustomValueEstimate']:\n",
    "    p99 = df[col].quantile(0.99)\n",
    "    outliers = (df[col] > p99).sum()\n",
    "    print(f'  {col}: {outliers:,} outliers above {p99:,.0f}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 7. Guiding Question 3: Temporal Trends"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df['TransactionMonth'] = pd.to_datetime(df['TransactionMonth'], errors='coerce')\n",
    "monthly = df.groupby(df['TransactionMonth'].dt.to_period('M')).agg(\n",
    "    AvgPremium=('TotalPremium','mean'),\n",
    "    AvgClaims=('TotalClaims','mean'),\n",
    "    ClaimFreq=('HasClaim','mean'),\n",
    "    LossRatio=('LossRatio','mean')\n",
    ").reset_index()\n",
    "monthly['TransactionMonth'] = monthly['TransactionMonth'].astype(str)\n",
    "\n",
    "fig, axes = plt.subplots(2, 1, figsize=(14, 8))\n",
    "fig.suptitle('Temporal Trends: Feb 2014 – Aug 2015', fontsize=14, fontweight='bold')\n",
    "\n",
    "axes[0].plot(monthly['TransactionMonth'], monthly['AvgPremium'], marker='o', label='Avg Premium', color='#2980b9')\n",
    "axes[0].plot(monthly['TransactionMonth'], monthly['AvgClaims'], marker='s', label='Avg Claims', color='#e74c3c')\n",
    "axes[0].set_ylabel('ZAR')\n",
    "axes[0].legend()\n",
    "axes[0].set_title('Average Premium and Claims Over Time')\n",
    "axes[0].tick_params(axis='x', rotation=45)\n",
    "\n",
    "axes[1].bar(monthly['TransactionMonth'], monthly['ClaimFreq']*100, color='#e67e22', alpha=0.8)\n",
    "axes[1].set_ylabel('Claim Frequency (%)')\n",
    "axes[1].set_title('Monthly Claim Frequency')\n",
    "axes[1].tick_params(axis='x', rotation=45)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/plot3_temporal.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()\n",
    "print('Plot 3 saved.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 8. Guiding Question 4: Vehicle Makes and Claims"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "make_claims = df.groupby('Make').agg(\n",
    "    AvgClaims=('TotalClaims','mean'),\n",
    "    Count=('PolicyID','count')\n",
    ").query('Count >= 100').sort_values('AvgClaims', ascending=False)\n",
    "\n",
    "print('Top 10 highest avg claim vehicle makes:')\n",
    "print(make_claims.head(10).round(2))\n",
    "print('\\nTop 10 lowest avg claim vehicle makes:')\n",
    "print(make_claims.tail(10).round(2))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 9. Plot 4: Correlation Matrix"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "num_cols = ['TotalPremium','TotalClaims','LossRatio','Margin','CustomValueEstimate',\n",
    "            'SumInsured','CalculatedPremiumPerTerm','Kilowatts','Cylinders']\n",
    "corr = df[num_cols].dropna().corr()\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(11, 8))\n",
    "mask = np.triu(np.ones_like(corr, dtype=bool))\n",
    "sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',\n",
    "            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 8})\n",
    "ax.set_title('Correlation Matrix — Key Financial & Vehicle Features', fontsize=13, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/plot4_correlation.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()\n",
    "print('Plot 4 saved.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 10. Plot 5: Outlier Detection"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(1, 3, figsize=(16, 5))\n",
    "fig.suptitle('Outlier Detection — Key Financial Variables', fontsize=13, fontweight='bold')\n",
    "\n",
    "for ax, col in zip(axes, ['TotalPremium','TotalClaims','CustomValueEstimate']):\n",
    "    data = df[col].dropna()\n",
    "    ax.boxplot(data[data.between(data.quantile(0.01), data.quantile(0.99))],\n",
    "               patch_artist=True, boxprops=dict(facecolor='#3498db', alpha=0.6))\n",
    "    ax.set_title(col)\n",
    "    ax.set_ylabel('ZAR')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/plot5_outliers.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()\n",
    "print('Plot 5 saved.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 11. EDA Summary\n\n**Key Findings:**\n\n1. **Overall Loss Ratio**: The portfolio loss ratio indicates overall profitability. Provinces with LR > 1.0 are loss-making and require premium adjustments.\n\n2. **Financial Distributions**: TotalClaims and TotalPremium are heavily right-skewed. Significant outliers exist above the 99th percentile and should be handled before modeling.\n\n3. **Temporal Trends**: Claim frequency and severity show variation over the 18-month period, with potential seasonal patterns worth investigating.\n\n4. **Vehicle Makes**: Significant variation in average claim amounts across vehicle makes provides evidence for make-based premium differentiation.\n\n5. **Correlations**: Strong positive correlation between TotalPremium and SumInsured confirms premium calculation logic. TotalClaims shows moderate correlation with vehicle value metrics."]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.13.2"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}